# Lekcija 32: Višeslojni perceptroni

Jedan neuron iz lekcije 31, koliko god pažljivo treniran, nije mogao nadmašiti slučajno pogađanje na skupu podataka s prstenom unutar diska &mdash; jedna linearna projekcija jednostavno nije dovoljno izražajna. Ova lekcija dodaje još jedan sloj: umjesto *jedne* projekcije iza koje slijedi prag, koristimo *dvije* projekcije s nelinearnošću između njih. To je sve. To je cijela ideja svake duboke mreže u ovom kolegiju &mdash; slagati više takvih slojeva.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Ponovno nerješivi problem

In [ ]:
rng = np.random.default_rng(0)
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X = np.vstack([inner, outer])
y = np.concatenate([np.zeros(60), np.ones(60)])

plt.scatter(*inner.T, s=15, label='inner (y=0)')
plt.scatter(*outer.T, s=15, label='outer (y=1)')
plt.legend(fontsize=8)
plt.gca().set_aspect('equal')
plt.title('No single linear projection separates these (Lessons 30-31)')
plt.show()

## Dvije projekcije, jedna nelinearnost

**Višeslojni perceptron (MLP)** povezuje *skrivenu* projekciju u novi prostor, nelinearnu **aktivaciju** primijenjenu po elementima i završnu linearnu projekciju (upravo neuron iz lekcije 31) nad *transformiranim* koordinatama:

$$h = \text{ReLU}(W_1 x + b_1), \qquad z = w_2^\top h + b_2, \qquad p = \sigma(z)$$

$\text{ReLU}(u) = \max(0, u)$ je najjednostavnija uobičajena aktivacija: linearna je svugdje osim na jednom prijelomu u nuli. Taj jedan prijelom, primijenjen neovisno na svaku jedinicu $h$, dovoljan je &mdash; skriveni sloj može savijati, presavijati i rastezati ulazni prostor tako da *linearna* granica u transformiranom prostoru odgovara izrazito nelinearnoj granici u izvornim koordinatama.

### Zašto je nelinearnost nužna

Bez ReLU-a &mdash; kada bi skriveni sloj ostao samo $h = W_1 x + b_1$ &mdash; dva sloja svela bi se na jednu linearnu projekciju:

$$z = w_2^\top(W_1 x + b_1) + b_2 = \underbrace{(w_2^\top W_1)}_{w'^\top} x + \underbrace{(w_2^\top b_1 + b_2)}_{b'}$$

To je ponovno upravo jedan neuron iz lekcije 31, samo s težinama dvaju slojeva pomnoženima u jedan $w'$ i $b'$. To je druga strana opažanja iz lekcije 30 da je množenje matrica skup projekcija: kompozicija dvaju množenja matrica *i dalje* je samo jedno množenje matrica, pa slaganje potpuno linearnih slojeva ne donosi ništa, bez obzira na njihov broj. ReLU-ov prijelom sprječava to svođenje &mdash; budući da nije linearna funkcija, $w_2^\top \text{ReLU}(W_1x+b_1)+b_2$ se više ne može prepisati kao neki jedan $w'^\top x + b'$.

In [ ]:
demo_rng = np.random.default_rng(3)
W1_demo = demo_rng.normal(size=(2, 4))
b1_demo = demo_rng.normal(size=4)
W2_demo = demo_rng.normal(size=(4, 1))
b2_demo = demo_rng.normal(size=1)

z_two_linear_layers = (X @ W1_demo + b1_demo) @ W2_demo + b2_demo   # no nonlinearity in between

W_combined = W1_demo @ W2_demo                # a single (2, 1) matrix
b_combined = b1_demo @ W2_demo + b2_demo      # a single offset
z_one_combined_layer = X @ W_combined + b_combined

print(f'max diff, two LINEAR layers vs. one combined layer: '
      f'{np.abs(z_two_linear_layers - z_one_combined_layer).max():.2e}')

## Povratno širenje pogreške kroz dva sloja

Lančano pravilo jednostavno se proširuje: signal pogreške $\partial L/\partial z$ iz lekcije 31 propagiramo unatrag kroz izlaznu projekciju kako bismo dobili $\partial L/\partial h$, a zatim kroz ReLU i skrivenu projekciju kako bismo dobili $\partial L/\partial W_1, \partial L/\partial b_1$. Svaki korak ili je „pomnoži transponiranom matricom težina” ili „pomnoži po elementima derivacijom aktivacije” &mdash; to je *sve* što povratno širenje pogreške radi, bez obzira na broj slojeva.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(np.float64)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = (a1 @ W2 + b2).ravel()
    p = sigmoid(z2)
    return p, (z1, a1, z2)

def backward(X, y, p, cache, W2):
    z1, a1, z2 = cache
    n = len(y)
    grad_z2 = ((p - y) / n).reshape(-1, 1)
    grad_W2 = a1.T @ grad_z2
    grad_b2 = grad_z2.sum(axis=0)
    grad_a1 = grad_z2 @ W2.T
    grad_z1 = grad_a1 * relu_deriv(z1)
    grad_W1 = X.T @ grad_z1
    grad_b1 = grad_z1.sum(axis=0)
    return grad_W1, grad_b1, grad_W2, grad_b2

### Provjera ispravnosti pomoću PyTorch autograda

In [ ]:
H = 4
init_rng = np.random.default_rng(8)
W1 = init_rng.normal(size=(2, H)) * 0.7
b1 = np.zeros(H)
W2 = init_rng.normal(size=(H, 1)) * 0.7
b2 = np.zeros(1)

p, cache = forward(X, W1, b1, W2, b2)
grad_W1, grad_b1, grad_W2, grad_b2 = backward(X, y, p, cache, W2)

X_t = torch.tensor(X)
y_t = torch.tensor(y)
W1_t = torch.tensor(W1, requires_grad=True)
b1_t = torch.tensor(b1, requires_grad=True)
W2_t = torch.tensor(W2, requires_grad=True)
b2_t = torch.tensor(b2, requires_grad=True)

z1_t = X_t @ W1_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = (a1_t @ W2_t + b2_t).squeeze(-1)
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z2_t, y_t)
loss_t.backward()

print(f'W1 max diff: {np.abs(grad_W1 - W1_t.grad.numpy()).max():.2e}')
print(f'W2 max diff: {np.abs(grad_W2 - W2_t.grad.numpy()).max():.2e}')
print(f'b1 max diff: {np.abs(grad_b1 - b1_t.grad.numpy()).max():.2e}')
print(f'b2 max diff: {np.abs(grad_b2 - b2_t.grad.numpy()).max():.2e}')

## Treniranje

In [ ]:
lr = 0.1
losses = []
for epoch in range(3000):
    p, cache = forward(X, W1, b1, W2, b2)
    eps = 1e-9
    losses.append(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)))
    grad_W1, grad_b1, grad_W2, grad_b2 = backward(X, y, p, cache, W2)
    W1 -= lr * grad_W1; b1 -= lr * grad_b1
    W2 -= lr * grad_W2; b2 -= lr * grad_b2

final_pred = (p > 0.5).astype(float)
print(f'final accuracy: {(final_pred == y).mean():.1%}  (single neuron, Lesson 31, managed 50%)')

plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training loss (2-layer MLP)')
plt.show()

## Prikaz projekcija skrivenog sloja

Svaka skrivena jedinica i sama je projekcija kao u lekciji 30: smjer $w_i$ (jedan stupac $W_1$) i pomak $b_i$ (jedan element $b_1$), koji zajedno definiraju rez $w_i \cdot x + b_i = 0$ s čije jedne strane ReLU postavlja vrijednosti na nulu. Uz samo 2 ulazne dimenzije, svih $H$ takvih rezova možemo nacrtati izravno preko izvornih podataka.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*X[y == 0].T, s=15, label='inner')
ax.scatter(*X[y == 1].T, s=15, label='outer')

colors = plt.cm.tab10.colors
for i in range(W1.shape[1]):
    w_i, b_i = W1[:, i], b1[i]
    # the cut {x : w_i.x + b_i = 0}, drawn through its closest point to the origin
    foot = -b_i * w_i / (w_i @ w_i)
    perp = np.array([-w_i[1], w_i[0]])
    p1, p2 = foot + 5 * perp, foot - 5 * perp
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=colors[i], linewidth=1.5, label=f'hidden unit {i}')
    ax.arrow(*foot, *(w_i / np.linalg.norm(w_i) * 0.5), head_width=0.1, color=colors[i], length_includes_head=True)

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.legend(fontsize=7, loc='upper right')
ax.set_title('Each hidden unit is one projection direction $w_i$,\ncutting the space along $w_i \\cdot x + b_i = 0$')
plt.tight_layout()
plt.show()

Svaki rez zasebno samo je pravac &mdash; nijedan pojedinačno ne razdvaja prsten od diska. No izlazni sloj kombinira svih $H$ njihovih rezultata (nakon ReLU-a), a nekoliko ravnih rezova raspoređenih oko prstena, kombiniranih odgovarajućim težinama, može dovoljno dobro aproksimirati zatvorenu krivulju da ga okruži. Upravo taj korak kombiniranja vizualiziramo u ostatku ovog odjeljka.

## Što je skriveni sloj zapravo učinio

Izlazni sloj *samo* je linearna projekcija (neuron iz lekcije 31) &mdash; ali djeluje na $h$, izlaz skrivenog sloja, a ne na izvorni $x$. Ako je skriveni sloj odradio svoj posao, *transformirane* točke već bi se trebale mnogo lakše razdvojiti pravcem. Budući da se $h$ ovdje nalazi u $\mathbb{R}^4$, koristimo PCA (lekcija 6) kako bismo ga prikazali u 2D.

In [ ]:
_, (_, hidden, _) = forward(X, W1, b1, W2, b2)

hidden_centered = hidden - hidden.mean(axis=0)
cov = np.cov(hidden_centered.T)
eigvals, eigvecs = np.linalg.eigh(cov)
top2_directions = eigvecs[:, -2:]
hidden_2d = hidden_centered @ top2_directions

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(*X[y == 0].T, s=15, label='inner')
axes[0].scatter(*X[y == 1].T, s=15, label='outer')
axes[0].set_aspect('equal')
axes[0].set_title('Original input space')
axes[0].legend(fontsize=7)

axes[1].scatter(*hidden_2d[y == 0].T, s=15, label='inner')
axes[1].scatter(*hidden_2d[y == 1].T, s=15, label='outer')
axes[1].set_title('Hidden representation\n(top 2 PCA directions of a 4D space)')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

Čak i ovaj 2D prikaz cijelog 4D skrivenog prostora, uz gubitak informacija, pokazuje dvije klase razmaknute u nešto blisko linearnoj razdvojivosti &mdash; velik napredak u odnosu na granicu od 74 %, najbolje što je *bilo koji* pravac mogao postići u izvornom prostoru (lekcija 30). Stvarni izlazni neuron radi u cijelom 4D skrivenom prostoru, gdje postiže točno 100 % točnosti. To je cijeli mehanizam dubokog učenja u malom: svaki sloj *preoblikuje prostor* tako da sljedećem sloju olakša posao, sve dok posao završnog sloja ne postane trivijalan &mdash; jedna linearna projekcija.

## Kapacitet modela: koliko je skrivenih jedinica dovoljno?

**Kapacitet** modela označava koliko bogatu obitelj funkcija može predstavljati. Ovdje ga izravno određuje $H$, širina skrivenog sloja: svaka skrivena jedinica daje jedan ravni rez (kao u lekciji 30), a izlazni sloj kombinira te jedinice. Uz premalo jedinica model jednostavno ne može predstavljati granicu potrebnu skupu podataka — nikakva količina treniranja ne može pronaći rješenje koje ne postoji. Mijenjajte $H$ i pri svakoj širini trenirajte od početka, uz nekoliko nasumičnih inicijalizacija, kako biste izravno vidjeli granicu mogućnosti.

In [ ]:
def train_and_eval(H, seed, epochs=3000, lr=0.1):
    init_rng = np.random.default_rng(seed)
    W1_s = init_rng.normal(size=(2, H)) * 0.7
    b1_s = np.zeros(H)
    W2_s = init_rng.normal(size=(H, 1)) * 0.7
    b2_s = np.zeros(1)
    for _ in range(epochs):
        p, cache = forward(X, W1_s, b1_s, W2_s, b2_s)
        gW1, gb1, gW2, gb2 = backward(X, y, p, cache, W2_s)
        W1_s -= lr * gW1; b1_s -= lr * gb1
        W2_s -= lr * gW2; b2_s -= lr * gb2
    p, _ = forward(X, W1_s, b1_s, W2_s, b2_s)
    return ((p > 0.5).astype(float) == y).mean()

widths = [1, 2, 3, 4, 6, 8, 16]
seeds = range(5)
mean_accs, min_accs, max_accs = [], [], []
for H_test in widths:
    accs = [train_and_eval(H_test, seed) for seed in seeds]
    mean_accs.append(np.mean(accs)); min_accs.append(np.min(accs)); max_accs.append(np.max(accs))
    print(f'H={H_test:>2}: mean acc={np.mean(accs):.1%}  (min={np.min(accs):.1%}, max={np.max(accs):.1%}, over {len(seeds)} seeds)')

plt.figure(figsize=(5.5, 3.5))
plt.plot(widths, mean_accs, '-o', label='mean')
plt.fill_between(widths, min_accs, max_accs, alpha=0.2, label='min-max range')
plt.xlabel('hidden units ($H$)'); plt.ylabel('final accuracy')
plt.title('Accuracy vs. capacity, across 5 random seeds each')
plt.legend(fontsize=8)
plt.show()

$H=1$ ostaje znatno ispod 100 % bez obzira na trajanje treniranja i početnu vrijednost generatora — to nije neuspjeh optimizacije, nego granica mogućnosti predstavljanja: jedna skrivena jedinica samo je jedan rez, pa gotovo točno nasljeđuje linearnu granicu od oko 74 % iz lekcije 30. $H=2$ je bolji, ali i dalje ograničen kapacitetom — nijedna od pet gornjih inicijalizacija ne doseže 100 %. $H=3$ je stvarna prijelomna točka: kapacitet za predstavljanje savršenog rješenja postoji (jedna gornja inicijalizacija doseže točno 100 %), ali hoće li ga treniranje pronaći ovisi o *početnoj vrijednosti* generatora (najava problema optimizacijskog krajolika iz lekcije 33).

Pri $H=4$ svaka inicijalizacija pouzdano doseže 100 %: kapacitet je dovoljan da mreža prestane biti usko grlo, a treniranje pouzdano pronalazi rješenje unutar tog kapaciteta. To je opći oblik krivulje kapaciteta — čvrsto ograničenje određeno onime što model *može* predstavljati, uz poteškoće optimizacije blizu granice tog ograničenja. Na istu se krivulju lekcija 35 vraća iz suprotnog smjera: ondje kapacitet ostaje velik, a *podataka* je malo, pa se „dovoljan kapacitet za predstavljanje pravog odgovora” pretvara u „ujedno dovoljan kapacitet za pamćenje pogrešnog odgovora”.

Širina nije jedina postavka. **Dubina** — slaganje više slojeva umjesto proširivanja jednog — druga je os i nije samo „prikrivena širina”: duboka, uska mreža može neke funkcije predstavljati mnogo kompaktnije nego plitka, široka (s manje ukupnih jedinica za istu izražajnost), ali uvodi vlastiti način neuspjeha. Lekcija 37 to izravno obrađuje: dovoljan broj slojeva može uzrokovati nestajanje gradijenata prije nego što stignu do najranijih slojeva, što se ne događa u plitkoj dvoslojnoj mreži ove lekcije.

### Zadaci

1. Gornje ispitivanje kapaciteta koristilo je ReLU. Ponovite ga uz `tanh` (zamijenite aktivaciju i njezinu derivaciju u `forward`/`backward` kroz cijeli `train_and_eval`, po uzoru na zadatak 2 u nastavku). Pojavljuju li se ista granica za $H=1$ i prijelaz s $H=3$ na $H=4$ ili glatka aktivacija mijenja položaj granice?
2. Svugdje zamijenite `relu`/`relu_deriv` s `tanh`/njezinom derivacijom ($1 - \tanh^2$) i ponovno trenirajte. Rješava li se problem i dalje? Usporedite oblik dobivene krivulje gubitka s ReLU varijantom.
3. Potpuno uklonite nelinearnost (zamijenite `relu(z1)` sa samo `z1` u `forward`, a `relu_deriv(z1)` poljem jedinica u `backward`). Potvrdite da mreža više ne može nadmašiti granicu od oko 50 % iz lekcije 31 &mdash; u skladu s gornjim argumentom svođenja na jednu projekciju, sada pokazanim na stvarnoj dinamici treniranja umjesto samo statičnom algebrom.